In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString
import warnings
warnings.filterwarnings('ignore')

In [38]:
class RouteProblematicAnalyzer:
    """
    Анализ проблемных участков УДС на маршрутах НГПТ
    Поля OSM: highway, surface, maxspeed, name
    """
    
    # Подходящие для магистрального НГПТ
    SUITABLE_HIGHWAY = {
        'primary', 'secondary', 'tertiary',
        'primary_link', 'secondary_link', 'tertiary_link'
    }
    
    # Проблемные классы дорог
    PROBLEMATIC_HIGHWAY = {
        'motorway', 'trunk', 'motorway_link', 'trunk_link',
        'residential', 'living_street', 'service', 'unclassified',
        'pedestrian', 'track', 'path', 'footway', 'cycleway', 'steps'
    }
    
    # Проблемные типы покрытия
    UNSUITABLE_SURFACE = {
        'unpaved', 'gravel', 'dirt', 'grass', 'sand', 'mud',
        'ground', 'earth', 'clay', 'compacted', 'fine_gravel',
        'pebblestone', 'rock', 'wood'
    }
    
    # Оценка полос по классу дороги
    ESTIMATED_LANES = {
        'primary': 4, 'secondary': 4, 'tertiary': 2,
        'residential': 2, 'living_street': 1, 'service': 1, 'unclassified': 2
    }
    
    def __init__(self, routes_gdf, osm_highway_gdf, crs_metric=32637):
        self.crs_metric = crs_metric
        self.routes = routes_gdf.to_crs(epsg=crs_metric).copy()
        self.roads = osm_highway_gdf.to_crs(epsg=crs_metric).copy()
        self.roads_sindex = self.roads.sindex
        
        print(f"Маршрутов: {len(self.routes)}")
        print(f"Сегментов УДС: {len(self.roads)}")
    
    def match_route_to_roads(self, route_geom, buffer_distance=25):
        route_buffer = route_geom.buffer(buffer_distance)
        possible_idx = list(self.roads_sindex.intersection(route_buffer.bounds))
        
        if not possible_idx:
            return gpd.GeoDataFrame()
        
        possible = self.roads.iloc[possible_idx].copy()
        mask = possible.geometry.intersects(route_buffer)
        matched = possible[mask].copy()
        
        if len(matched) == 0:
            return gpd.GeoDataFrame()
        
        matched['matched_length'] = matched.geometry.intersection(route_buffer).length
        matched = matched[matched['matched_length'] > 5].copy()
        
        return matched
    
    def evaluate_segment(self, row):
        problems = {
            'unsuitable_class': False,
            'unsuitable_surface': False,
            'low_speed': False,
            'sharp_turns': False,
            'narrow_estimated': False
        }
        
        highway = str(row.get('highway', '')).lower().strip()
        
        # 1. Класс дороги
        if highway in self.PROBLEMATIC_HIGHWAY:
            problems['unsuitable_class'] = True
        elif highway not in self.SUITABLE_HIGHWAY and highway != '':
            problems['unsuitable_class'] = True
        
        # 2. Оценка ширины
        est_lanes = self.ESTIMATED_LANES.get(highway, 2)
        if est_lanes < 4:
            problems['narrow_estimated'] = True
        
        # 3. Покрытие
        surface = str(row.get('surface', '')).lower().strip()
        if surface and surface in self.UNSUITABLE_SURFACE:
            problems['unsuitable_surface'] = True
        
        # 4. Скорость
        maxspeed = self._parse_maxspeed(row.get('maxspeed'))
        if maxspeed is not None and maxspeed < 40:
            problems['low_speed'] = True
        elif highway in {'living_street', 'pedestrian'}:
            problems['low_speed'] = True
        
        # 5. Острые повороты
        geom = row.get('geometry')
        if geom is not None and isinstance(geom, LineString):
            problems['sharp_turns'] = self._has_sharp_turns(geom, threshold=60)
        
        # Итог
        serious = (problems['unsuitable_class'] or 
                   problems['unsuitable_surface'] or 
                   problems['low_speed'])
        problems['is_problematic'] = serious
        
        return problems
    
    def _parse_maxspeed(self, value):
        if pd.isna(value) or value is None:
            return None
        try:
            val_str = str(value).lower()
            val_str = val_str.replace('km/h', '').replace('kmh', '').replace('mph', '').strip()
            if val_str in ('walk', 'ru:living_street'):
                return 20
            if val_str == 'ru:urban':
                return 60
            if val_str == 'ru:rural':
                return 90
            return int(float(val_str))
        except:
            return None
    
    def _has_sharp_turns(self, geom, threshold=60):
        coords = list(geom.coords)
        if len(coords) < 3:
            return False
        for i in range(1, len(coords) - 1):
            angle = self._angle_between(coords[i-1], coords[i], coords[i+1])
            if angle < (180 - threshold):
                return True
        return False
    
    def _angle_between(self, p1, p2, p3):
        v1 = np.array([p1[0] - p2[0], p1[1] - p2[1]])
        v2 = np.array([p3[0] - p2[0], p3[1] - p2[1]])
        norm = np.linalg.norm(v1) * np.linalg.norm(v2)
        if norm < 1e-10:
            return 180
        cos_angle = np.clip(np.dot(v1, v2) / norm, -1, 1)
        return np.degrees(np.arccos(cos_angle))
    
    def calculate_for_route(self, route_id, route_geom, buffer_distance=25):
        matched = self.match_route_to_roads(route_geom, buffer_distance)
        L_route = route_geom.length
        
        if len(matched) == 0:
            return {
                'route_id': route_id,
                'L': round(L_route, 1),
                'L_matched': 0,
                'L_probl': 0,
                'b': None,
                'b_percent': None,
                'n_segments': 0,
                'coverage': 0,
                'highway_types': '',
                'highway_breakdown': '',
                'L_unsuitable_class': 0,
                'L_unsuitable_surface': 0,
                'L_low_speed': 0,
                'L_sharp_turns': 0,
                'L_narrow_estimated': 0,
                'warning': 'Не удалось привязать к УДС'
            }
        
        # Оценка каждого сегмента
        results = []
        for idx, row in matched.iterrows():
            problems = self.evaluate_segment(row)
            results.append({
                'length': row['matched_length'],
                'is_problematic': problems['is_problematic'],
                'highway': str(row.get('highway', '')).lower().strip(),
                **problems
            })
        
        df = pd.DataFrame(results)
        
        L_matched = df['length'].sum()
        L_probl = df[df['is_problematic']]['length'].sum()
        b = L_probl / L_matched if L_matched > 0 else 0
        
        # Детализация по типам проблем
        problem_stats = {}
        for col in ['unsuitable_class', 'unsuitable_surface', 'low_speed', 
                    'sharp_turns', 'narrow_estimated']:
            prob_len = df[df[col] == True]['length'].sum()
            problem_stats[col] = round(prob_len, 1)
        
        # Статистика по highway
        highway_stats = df.groupby('highway')['length'].sum().sort_values(ascending=False)
        
        # Список типов highway
        highway_types = ', '.join(highway_stats.index.tolist())
        
        # Детализация: тип (длина, доля)
        highway_breakdown = '; '.join([
            f"{hw}: {length:.0f}м ({length/L_matched*100:.1f}%)" 
            for hw, length in highway_stats.items()
        ])
        
        # Длины по каждому типу highway (отдельные колонки)
        highway_lengths = {f'hw_{hw}': round(length, 1) 
                          for hw, length in highway_stats.items()}
        
        return {
            'route_id': route_id,
            'L': round(L_route, 1),
            'L_matched': round(L_matched, 1),
            'L_probl': round(L_probl, 1),
            'b': round(b, 4),
            'b_percent': round(b * 100, 1),
            'n_segments': len(matched),
            'coverage': round(L_matched / L_route, 2) if L_route > 0 else 0,
            'highway_types': highway_types,
            'highway_breakdown': highway_breakdown,
            **highway_lengths,
            **{f'L_{k}': v for k, v in problem_stats.items()}
        }
    
    def calculate_all(self, route_id_col='route_id', buffer_distance=25):
        results = []
        total = len(self.routes)
        
        for i, (idx, row) in enumerate(self.routes.iterrows()):
            route_id = row.get(route_id_col, idx)
            
            if (i + 1) % 10 == 0:
                print(f"Обработано {i+1}/{total}...")
            
            result = self.calculate_for_route(route_id, row.geometry, buffer_distance)
            results.append(result)
        
        print(f"Готово! Обработано {total} маршрутов")
        return pd.DataFrame(results)


def summarize_highway_usage(results):
    """Сводка использования типов highway по всем маршрутам"""
    
    SUITABLE = {'primary', 'secondary', 'tertiary',
                'primary_link', 'secondary_link', 'tertiary_link'}
    
    hw_cols = [c for c in results.columns if c.startswith('hw_')]
    
    if not hw_cols:
        print("Нет данных по highway")
        return None
    
    summary = []
    for col in hw_cols:
        hw_type = col.replace('hw_', '')
        total_length = results[col].sum()
        routes_count = (results[col] > 0).sum()
        summary.append({
            'highway': hw_type,
            'total_length_m': round(total_length, 0),
            'total_length_km': round(total_length / 1000, 2),
            'routes_count': routes_count,
            'is_suitable': hw_type in SUITABLE
        })
    
    summary_df = pd.DataFrame(summary).sort_values('total_length_m', ascending=False)
    
    print("\n" + "="*60)
    print("ИСПОЛЬЗОВАНИЕ ТИПОВ ДОРОГ")
    print("="*60)
    print(summary_df.to_string(index=False))
    
    # Итого
    suitable = summary_df[summary_df['is_suitable']]['total_length_m'].sum()
    unsuitable = summary_df[~summary_df['is_suitable']]['total_length_m'].sum()
    total = suitable + unsuitable
    
    if total > 0:
        print(f"\nПодходящие (primary/secondary/tertiary): {suitable/1000:.1f} км ({suitable/total*100:.1f}%)")
        print(f"Проблемные: {unsuitable/1000:.1f} км ({unsuitable/total*100:.1f}%)")
    
    return summary_df



In [39]:
# =============================================================================
# ЗАПУСК РАСЧЁТА
# =============================================================================

# 1. Загрузка данных
routes = gpd.read_file(r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\routes\yandex_izhevsk_28409.gpkg", layer='yandex_routes')     # <-- ваш файл маршрутов
routes = routes[(routes['ya_vhc_type'] != 'tramway') & (routes['main_route'] == 'true')]
routes = routes.rename(columns={'ya_route_id': 'route_id'})

roads = gpd.read_file(r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\highway_cleaned.gpkg")      # <-- ваш файл highway из OSM
roads = roads[~roads['highway'].isin(['footway', 'path'])]

In [62]:
# 2. Создание анализатора
analyzer = RouteProblematicAnalyzer(
    routes_gdf=routes,
    osm_highway_gdf=roads,
    crs_metric=32639  # <-- EPSG для вашего региона
)

# 3. Расчёт
results = analyzer.calculate_all(
    route_id_col='route_id',  # <-- название поля с ID маршрута
    buffer_distance=25
)

# 4. Вывод результатов
print("\n" + "="*70)
print("РЕЗУЛЬТАТЫ ПО МАРШРУТАМ")
print("="*70)
print(results[['route_id', 'L', 'L_probl', 'b_percent', 'coverage', 'highway_types']].to_string())

# 5. Статистика
print("\n" + "="*70)
print("ОБЩАЯ СТАТИСТИКА")
print("="*70)
valid = results[results['b'].notna()]
print(f"Маршрутов с данными: {len(valid)} из {len(results)}")
print(f"Средняя доля проблемных: {valid['b_percent'].mean():.1f}%")
print(f"Медиана: {valid['b_percent'].median():.1f}%")
print(f"Мин: {valid['b_percent'].min():.1f}%")
print(f"Макс: {valid['b_percent'].max():.1f}%")

# 6. Сводка по highway
highway_summary = summarize_highway_usage(results)

# # 7. Сохранение
# results.to_csv("route_problematic_share.csv", index=False, encoding='utf-8-sig')
# if highway_summary is not None:
#     highway_summary.to_csv("highway_summary.csv", index=False, encoding='utf-8-sig')

print("\nСохранено:")
print("  - route_problematic_share.csv")
print("  - highway_summary.csv")

Маршрутов: 73
Сегментов УДС: 23287
Обработано 10/73...
Обработано 20/73...
Обработано 30/73...
Обработано 40/73...
Обработано 50/73...
Обработано 60/73...
Обработано 70/73...
Готово! Обработано 73 маршрутов

РЕЗУЛЬТАТЫ ПО МАРШРУТАМ
      route_id        L  L_probl  b_percent  coverage                                                                                                                                                            highway_types
0   2161483940   6920.2   4200.7       32.3      1.88                                                                                  primary, service, residential, secondary, unclassified, tertiary, steps, secondary_link
1   2161483946   7090.7   4163.2       31.5      1.87                                                                                  primary, service, residential, secondary, unclassified, tertiary, steps, secondary_link
2   2161483952  12880.2   8674.3       30.7      2.19                primary, secondary, service, r

In [63]:
results

,route_id,L,L_matched,L_probl,b,b_percent,n_segments,coverage,highway_types,highway_breakdown,...,L_low_speed,L_sharp_turns,L_narrow_estimated,hw_cycleway,hw_primary_link,hw_pedestrian,hw_tertiary_link,hw_living_street,hw_track,hw_construction
0,2161483940,6920.2,13011.3,4200.7,0.3228,32.3,174,1.88,"primary, service, residential, secondary, uncl...",primary: 8297м (63.8%); service: 2596м (19.9%)...,...,180.4,0.0,4292.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2161483946,7090.7,13230.9,4163.2,0.3147,31.5,183,1.87,"primary, service, residential, secondary, uncl...",primary: 8370м (63.3%); service: 2503м (18.9%)...,...,136.1,0.0,4372.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2161483952,12880.2,28268.8,8674.3,0.3069,30.7,417,2.19,"primary, secondary, service, residential, tert...",primary: 10321м (36.5%); secondary: 7698м (27....,...,1980.5,0.0,10250.0,532.3,243.3,26.6,14.0,5.0,NaN,NaN
3,2161483955,13075.1,29263.3,9156.2,0.3129,31.3,415,2.24,"primary, secondary, service, cycleway, residen...",primary: 10526м (36.0%); secondary: 7993м (27....,...,1909.6,0.0,10744.1,2158.8,275.8,26.7,14.0,136.9,NaN,NaN
4,2161484020,8183.6,13956.1,7709.5,0.5524,55.2,179,1.71,"residential, primary, tertiary, service, secon...",residential: 4680м (33.5%); primary: 3032м (21...,...,371.4,0.0,9963.1,381.7,NaN,NaN,NaN,NaN,10.2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,3357137945,12283.4,24569.8,8183.4,0.3331,33.3,379,2.00,"primary, secondary, service, tertiary, cyclewa...",primary: 6259м (25.5%); secondary: 6190м (25.2...,...,2262.4,0.0,12120.3,1891.9,23.4,NaN,191.8,NaN,21.9,NaN
69,3764809050,12435.2,24611.6,8006.7,0.3253,32.5,387,1.98,"primary, secondary, service, tertiary, cyclewa...",primary: 6603м (26.8%); secondary: 6168м (25.1...,...,2186.7,0.0,11839.7,1875.3,23.4,NaN,155.4,NaN,53.1,NaN
70,2161484058,26714.4,30307.4,9238.0,0.3048,30.5,408,1.13,"secondary, service, primary, tertiary, residen...",secondary: 11693м (38.6%); service: 7450м (24....,...,3222.5,0.0,11348.5,43.5,45.3,28.2,107.3,48.9,NaN,NaN
71,6063076059,23419.3,45898.3,12330.4,0.2686,26.9,637,1.96,"primary, secondary, service, residential, tert...",primary: 18131м (39.5%); secondary: 12990м (28...,...,3256.2,0.0,14777.3,365.0,89.9,28.2,642.9,77.7,NaN,NaN


In [64]:
results_m = results.merge(
    routes[['route_id', 'route_name_with_vhc_prefix', 'ya_line_id']],
    on='route_id',
    how='left'
)

In [65]:
# Переупорядочиваем столбцы
results_m = results_m[['route_id', 'ya_line_id', 'route_name_with_vhc_prefix', 'L', 'L_probl', 'b','b_percent']]
results_m = results_m.sort_values('b_percent', ascending=False).reset_index(drop=True)

results_m

,route_id,ya_line_id,route_name_with_vhc_prefix,L,L_probl,b,b_percent
0,3456270425,2161483926,bus_15,8155.3,8451.2,0.5876,58.8
1,3973084340,3079437094,bus_15к,6676.2,6378.6,0.5694,56.9
2,2161484020,2161483926,bus_15,8183.6,7709.5,0.5524,55.2
3,3973086270,3079437094,bus_15к,6788.5,5631.2,0.5165,51.7
4,2161483990,2161483909,bus_7,6876.3,6565.0,0.4908,49.1
...,...,...,...,...,...,...,...
68,3767018110,2161483925,trolleybus_14,18569.5,8807.3,0.2185,21.8
69,2172348372,2161483979,trolleybus_2,12299.7,5789.3,0.2170,21.7
70,2161484082,2161483977,minibus_68,21659.8,9654.5,0.2135,21.4
71,4206764453,2161483928,bus_16,13476.5,4999.5,0.1860,18.6


In [67]:
route_to_line = results_m.groupby('ya_line_id').agg(
    name              = ('route_name_with_vhc_prefix',            'first'),
    num_variants      = ('route_id',        'nunique'),     # сколько вариантов у маршрута
    L       = ('L',      'mean'),        # суммарное кол-во остановок
    L_probl    = ('L_probl',   'mean'),        # суммарное кол-во пересадочных
    b_percent_mean = ('b_percent', 'mean'),       # среднее по вариантам
    b_percent_min  = ('b_percent', 'min'),
    b_percent_max  = ('b_percent', 'max'),
).reset_index()

# # Взвешенная доля (по числу остановок) — обычно корректнее простого среднего
# route_to_line['transfer_share_weighted'] = (
#     route_to_line['transfer_stops'] / route_to_line['total_stops']
# ).round(3) * 100

route_to_line = route_to_line.sort_values('b_percent_mean', ascending=False).reset_index(drop=True)

route_to_line

,ya_line_id,name,num_variants,L,L_probl,b_percent_mean,b_percent_min,b_percent_max
0,2161483926,bus_15,2,8169.45,8080.35,57.00,55.2,58.8
1,3079437094,bus_15к,2,6732.35,6004.90,54.30,51.7,56.9
2,5166585596,minibus_66,2,17175.00,15008.60,47.90,47.7,48.1
3,2161483909,bus_7,2,7008.65,6339.00,46.30,43.5,49.1
4,2161483931,bus_19,2,13445.60,11481.45,42.85,41.5,44.2
5,2161483973,bus_56,2,37953.90,10424.80,39.40,39.4,39.4
6,2161483970,minibus_52,1,24436.20,14517.50,35.40,35.4,35.4
7,2161483954,bus_28,2,12827.85,10793.30,35.15,34.4,35.9
8,2161483988,bus_26,1,39463.20,15933.70,35.10,35.1,35.1
9,2161483912,bus_73,2,8457.70,5249.50,34.40,32.2,36.6


In [69]:
route_to_line.to_parquet(r'X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\routes\routes_wuds.parquet')